# Fase 5 — verificar antes de responder

O agente continua usando ferramentas, RAG e memória. Esta fase verifica a candidata antes de exibir e gravar o turno. Regras determinísticas conferem valores; um juiz LLM distingue sustentação, contradição e ausência de suporte. Isso é uma adaptação por LLM, não um classificador NLI especializado.

Sem API, este notebook mostra **fixtures identificadas e checagens locais**, sem inventar notas do juiz. Os mesmos módulos são usados na CLI.

In [1]:
# IMPORTAÇÃO: o notebook usa o pacote .py do projeto, sem duplicar o verificador.
from pathlib import Path
import sys, os
projeto = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '02_codigo/src').is_dir())
sys.path.insert(0, str(projeto / '02_codigo/src'))
from auto_super_parts.demonstracao_verificacao import demonstrar
from auto_super_parts.verificacao import criar_modelo_verificacao
modo = os.environ.get('FASE05_MODO', 'simulado')
print('Modo:', modo, '| fixtures são candidatas escritas para teste, não respostas reais do atendente.')

Modo: simulado | fixtures são candidatas escritas para teste, não respostas reais do atendente.


## Como a decisão é tomada

A pergunta, os retornos atuais das ferramentas e o contexto declarado recebem IDs. O juiz cita os IDs e avalia trechos literais da resposta. Python rejeita parecer incompleto, fonte inexistente e valor divergente. Perfil e falas antigas não provam preço atual.

Se reprovada, a candidata recebe no máximo um reparo e um novo julgamento. Se não passar ou se o juiz falhar, sai uma limitação segura. A memória recebe somente a resposta final. Uma oferta de consulta não equivale a responder.

In [2]:
# MODO EXPLÍCITO: sem API não executa juiz nem produz uma métrica semântica.
modelo = criar_modelo_verificacao() if modo == 'real' else None
resultado = demonstrar(modelo)
print(resultado['origem'])
print(resultado['limite'])

fixtures manuais; juiz NAO executado
Gabarito manual e comparação binária não comprovam acurácia geral nem equivalência a classificador NLI.


In [3]:
# EVIDÊNCIAS: separe gabarito das fixtures, regras locais e julgamento realmente executado.
from IPython.display import HTML, display
from html import escape
linhas = []
for c in resultado['casos']:
    a = c['avaliacao_juiz']
    linhas.append('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in [
        c['id'], c['esperado_aprovada'], c['classe_esperada'] or 'resposta efetiva',
        len(c['regras_locais']), a['aprovada'] if a else 'NÃO EXECUTADO', c['conferiu_gabarito']]) + '</tr>')
display(HTML('<table><tr><th>Fixture</th><th>Aceite esperado</th><th>Classe esperada</th><th>Alertas locais</th><th>Juiz</th><th>Conferiu gabarito</th></tr>' + ''.join(linhas) + '</table>'))

Fixture,Aceite esperado,Classe esperada,Alertas locais,Juiz,Conferiu gabarito
orcamento_correto,True,sustentada,0,NÃO EXECUTADO,None
total_incorreto,False,contradita,2,NÃO EXECUTADO,None
saldo_insuficiente,False,contradita,0,NÃO EXECUTADO,None
garantia_inventada,False,sem_suporte,0,NÃO EXECUTADO,None
lacuna_explicada,True,sustentada,0,NÃO EXECUTADO,None
cartao_contrario,False,contradita,0,NÃO EXECUTADO,None
oferta_nao_responde,False,resposta efetiva,0,NÃO EXECUTADO,None
precos_trocados,False,contradita,0,NÃO EXECUTADO,None
homologacao_inventada,False,contradita,0,NÃO EXECUTADO,None
injecao_na_candidata,False,contradita,0,NÃO EXECUTADO,None


## Limites e execução completa

Números existentes podem estar associados à peça errada; essa comparação exige o juiz semântico. O juiz também pode errar. Conferir 14 fixtures não comprova qualidade universal, nem produz relatório LangSmith. A classe sem suporte não deve virar contradição quando o dado é apenas desconhecido.

No terminal, a partir do projeto:

```powershell
python -B 02_codigo/validar_verificacao.py
python -B 02_codigo/validar_verificacao.py --real
python -B 02_codigo/src/auto_super_parts/atendimento_verificado.py
```

O modo real avalia as fixtures e executa integração em banco temporário: uma falha controlada (candidata injetada, juiz/reparo reais) e dois atendimentos com modelo real. A ausência de chave não autoriza inventar saídas. Decisões e limitações ficam no guia e nos registros do projeto.